In [ ]:
# =====================================================================
# CELLA IMPORTAZIONI E ATTIVAZIONE GPU APPLE SILICON (M1)
# =====================================================================
import os
import glob
import logging
import gc  # <--- IMPORTANTE: serve per svuotare la RAM satura
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from keras.models import load_model

# Silenziamo i log di sistema inutili
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
logging.getLogger('tensorflow').setLevel(logging.ERROR)

import tensorflow as tf
tf.get_logger().setLevel('ERROR')
tf.autograph.set_verbosity(0)

# CONTROLLO E ABILITAZIONE GPU MAC M1 (MPS - Metal Performance Shaders)
dispositivi_gpu = tf.config.list_physical_devices('GPU')
if dispositivi_gpu:
    print(f"Ottimo! GPU M1 Rilevata correttamente: {dispositivi_gpu}")
    # Nota: TensorFlow su Mac gestisce automaticamente l'allocazione su MPS
else:
    print("Nessuna GPU rilevata. Se hai un Mac M1, assicurati di aver installato 'tensorflow-metal'")

from keras import layers, models, losses
from keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

In [3]:
# =====================================================================
# CELLA IMPORTAZIONI LIBRERIE PER MAC
# =====================================================================
import os
import glob
import gc
import logging
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from keras.models import load_model

# Silenziamo i log inutili del Mac
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # Blocca tutto tranne gli errori fatali
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0' 
logging.getLogger('tensorflow').setLevel(logging.ERROR)

# NOTA: Rimosso 'TF_CUDNN_USE_AUTOTUNE' perché sul tuo Mac non serve

import tensorflow as tf

# Altri silenziatori di log 
tf.get_logger().setLevel('ERROR')
tf.autograph.set_verbosity(0)

# =====================================================================
# MODIFICA 1: DISABILITARE LA GPU PER EVITARE I GRADIENTI NaN (ESPLOSIONE DELLA LOSS)
# =====================================================================
# Diciamo a TensorFlow di "nascondere" la GPU M1 (gestita da tensorflow-metal).
tf.config.set_visible_devices([], 'GPU')

# Riga di controllo per essere sicuri al 100% che abbia funzionato
print("Dispositivi di calcolo attivi:", tf.config.get_visible_devices())
# =====================================================================

# Import di Keras (lasciati identici a quelli del tuo collega)
from keras import layers, models, losses
from keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

Dispositivi di calcolo attivi: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


In [ ]:
# ==============================================================================
# DATA ENGINE V11 (Memory Optimized: DBF a 12 angoli + Garbage Collection)
# ==============================================================================
def load_and_process_all_files(file_list, alpha=0.20, num_angles=12): # <--- Ridotto a 12 angoli per salvare la RAM
    X_all, Y_all = [], []
    print(f"Inizio DBF ottimizzato ed EMA Decluttering di {len(file_list)} file...")
    
    # Pre-calcolo della matrice dei pesi W per il Beamforming
    theta = np.linspace(-np.pi/3, np.pi/3, num_angles)
    n_antennas_array = np.arange(3)
    W = np.exp(-1j * np.pi * np.outer(np.sin(theta), n_antennas_array))
    
    for i, file_path in enumerate(file_list):
        data = np.load(file_path)
        raw_iq = data['radar_cir_iq']   
        people_xy = data['people_xy']   
        people_mask = data['people_mask'] 
        T = raw_iq.shape[0]             
        
        # Convertiamo I/Q in numeri complessi
        complex_iq = raw_iq[..., 0] + 1j * raw_iq[..., 1]
        
        # Trasposizione e Digital Beamforming
        iq_transposed = np.transpose(complex_iq, (0, 1, 3, 2))
        dbf_result = np.dot(iq_transposed, W.T)
        
        # Estrazione della Magnitudo ed eliminazione asse complessi
        mag = np.abs(dbf_result)
        mag_img = np.transpose(mag, (0, 2, 3, 1)).astype(np.float32)
        
        # EMA Decluttering
        decluttered = np.zeros_like(mag_img)
        bg = np.copy(mag_img[0])
        for t in range(T):
            bg = alpha * mag_img[t] + (1 - alpha) * bg
            decluttered[t] = np.abs(mag_img[t] - bg)
            
        # Normalizzazione locale [0, 1] per stabilità
        max_val = np.max(decluttered)
        min_val = np.min(decluttered)
        decluttered = (decluttered - min_val) / (max_val - min_val + 1e-8)
        
        # Target delle coordinate
        flat_coords = people_xy.reshape(T, 8)
        combined_target = np.concatenate([flat_coords, people_mask], axis=1)

        X_all.append(decluttered)
        Y_all.append(combined_target)
        
        print(f"Processato file {i+1}/{len(file_list)}. RAM Liberata.")
        
        # --- PULIZIA AGGRESSIVA DELLA MEMORIA AD OGNI ITERAZIONE ---
        del data, raw_iq, complex_iq, iq_transposed, dbf_result, mag, mag_img, decluttered
        gc.collect() # Forziamo il Mac a liberare la RAM fisica immediatamente

    X = np.concatenate(X_all, axis=0).astype(np.float32)
    Y = np.concatenate(Y_all, axis=0).astype(np.float32)
    return X, Y

# Split del Dataset (Inalterato)
val_indices = [23, 20, 0, 13, 9] 
train_indices = [22, 16, 17, 18, 19, 21, 1, 2, 3, 4, 12, 14, 15, 5, 6, 7, 8, 10, 11]

tutti_i_file = glob.glob("dataset/*.npz")
train_files = [f for f in tutti_i_file if int(os.path.basename(f).replace("window_", "").replace(".npz", "")) in train_indices]
val_files = [f for f in tutti_i_file if int(os.path.basename(f).replace("window_", "").replace(".npz", "")) in val_indices]

print("\n--- CARICAMENTO TRAINING SET ---")
X_train, Y_train = load_and_process_all_files(train_files)

print("\n--- CARICAMENTO VALIDATION SET ---")
X_val, Y_val = load_and_process_all_files(val_files)

print("\n==================================================")
print(f"DATI PRONTI IN RAM! Nuova shape di input: {X_train.shape[1:]}")
print("==================================================")

In [4]:
# ==============================================================================
# DATA ENGINE V12 (Fase 1 & 2: EMA su I/Q complessi + Normalizzazione Coordinate)
# ==============================================================================
def load_and_process_all_files(file_list, alpha=0.20, num_angles=12):
    X_all, Y_all = [], []
    print(f"Inizio DBF ottimizzato ed EMA Decluttering di {len(file_list)} file...")
    
    # Pre-calcolo della matrice W per il Beamforming
    # Assumiamo distanza tra antenne (d) = mezza lunghezza d'onda (lambda/2)
    d_lambda = 0.5 
    theta = np.linspace(-np.pi/3, np.pi/3, num_angles)
    n_antennas_array = np.arange(3)
    W = np.exp(-1j * 2 * np.pi * d_lambda * np.outer(np.sin(theta), n_antennas_array))
    
    for i, file_path in enumerate(file_list):
        data = np.load(file_path)
        raw_iq = data['radar_cir_iq']   
        people_xy = data['people_xy']   
        people_mask = data['people_mask'] 
        T = raw_iq.shape[0]             
        
        # 1. Convertiamo I/Q in numeri complessi
        complex_iq = raw_iq[..., 0] + 1j * raw_iq[..., 1]
        
        # 2. EMA Decluttering SUI NUMERI COMPLESSI (Fase 1)
        decluttered_iq = np.zeros_like(complex_iq)
        bg = np.copy(complex_iq[0])
        for t in range(T):
            bg = alpha * complex_iq[t] + (1 - alpha) * bg
            decluttered_iq[t] = complex_iq[t] - bg
            
        # 3. Trasposizione e Digital Beamforming (sui dati puliti)
        iq_transposed = np.transpose(decluttered_iq, (0, 1, 3, 2))
        dbf_result = np.dot(iq_transposed, W.T)
        
        # 4. Estrazione della Magnitudo
        mag = np.abs(dbf_result)
        mag_img = np.transpose(mag, (0, 2, 3, 1)).astype(np.float32)
        
        # 5. Normalizzazione locale [0, 1] dell'immagine radar
        max_val = np.max(mag_img)
        min_val = np.min(mag_img)
        mag_img = (mag_img - min_val) / (max_val - min_val + 1e-8)
        
        # --- FASE 2: NORMALIZZAZIONE DELLE COORDINATE [0, 1] ---
        # Stanza: X = 4.8m, Y = 7.2m
        norm_xy = np.copy(people_xy)
        norm_xy[:, :, 0] = norm_xy[:, :, 0] / 4.8  # Normalizza X
        norm_xy[:, :, 1] = norm_xy[:, :, 1] / 7.2  # Normalizza Y
        
        flat_coords = norm_xy.reshape(T, 8)
        combined_target = np.concatenate([flat_coords, people_mask], axis=1)

        X_all.append(mag_img)
        Y_all.append(combined_target)
        
        print(f"Processato file {i+1}/{len(file_list)}. RAM Liberata.")
        
        # Pulizia della memoria
        del data, raw_iq, complex_iq, decluttered_iq, iq_transposed, dbf_result, mag, mag_img, norm_xy
        gc.collect()

    X = np.concatenate(X_all, axis=0).astype(np.float32)
    Y = np.concatenate(Y_all, axis=0).astype(np.float32)
    return X, Y

# Split del Dataset (Inalterato)
val_indices = [23, 20, 0, 13, 9] 
train_indices = [22, 16, 17, 18, 19, 21, 1, 2, 3, 4, 12, 14, 15, 5, 6, 7, 8, 10, 11]

tutti_i_file = glob.glob("dataset/*.npz")
train_files = [f for f in tutti_i_file if int(os.path.basename(f).replace("window_", "").replace(".npz", "")) in train_indices]
val_files = [f for f in tutti_i_file if int(os.path.basename(f).replace("window_", "").replace(".npz", "")) in val_indices]

print("\n--- CARICAMENTO TRAINING SET ---")
X_train, Y_train = load_and_process_all_files(train_files)

print("\n--- CARICAMENTO VALIDATION SET ---")
X_val, Y_val = load_and_process_all_files(val_files)

print("\n==================================================")
print(f"DATI PRONTI IN RAM! Nuova shape di input: {X_train.shape[1:]}")
print("==================================================")


--- CARICAMENTO TRAINING SET ---
Inizio DBF ottimizzato ed EMA Decluttering di 19 file...
Processato file 1/19. RAM Liberata.
Processato file 2/19. RAM Liberata.
Processato file 3/19. RAM Liberata.
Processato file 4/19. RAM Liberata.
Processato file 5/19. RAM Liberata.
Processato file 6/19. RAM Liberata.
Processato file 7/19. RAM Liberata.
Processato file 8/19. RAM Liberata.
Processato file 9/19. RAM Liberata.
Processato file 10/19. RAM Liberata.
Processato file 11/19. RAM Liberata.
Processato file 12/19. RAM Liberata.
Processato file 13/19. RAM Liberata.
Processato file 14/19. RAM Liberata.
Processato file 15/19. RAM Liberata.
Processato file 16/19. RAM Liberata.
Processato file 17/19. RAM Liberata.
Processato file 18/19. RAM Liberata.
Processato file 19/19. RAM Liberata.

--- CARICAMENTO VALIDATION SET ---
Inizio DBF ottimizzato ed EMA Decluttering di 5 file...
Processato file 1/5. RAM Liberata.
Processato file 2/5. RAM Liberata.
Processato file 3/5. RAM Liberata.
Processato file 4/

In [5]:
# =====================================================================
# BULGARIAN SQUAT PER MAC
# =====================================================================
import itertools
PERM_INDICES = tf.constant(list(itertools.permutations([0, 1, 2, 3])), dtype=tf.int32)

def hungarian_total_loss(y_true, y_pred):
    y_true_coords = tf.reshape(y_true[:, :8], (-1, 4, 2))
    y_pred_coords = tf.reshape(y_pred[:, :8], (-1, 4, 2))
    y_true_mask = tf.reshape(y_true[:, 8:], (-1, 4, 1))
    y_pred_mask = tf.reshape(y_pred[:, 8:], (-1, 4, 1))

    y_pred_coords_perm = tf.gather(y_pred_coords, PERM_INDICES, axis=1) 
    y_pred_mask_perm = tf.gather(y_pred_mask, PERM_INDICES, axis=1)

    y_true_coords_exp = tf.expand_dims(y_true_coords, 1) 
    y_true_mask_exp = tf.expand_dims(y_true_mask, 1)

    sq_diff = tf.square(y_true_coords_exp - y_pred_coords_perm)
    coords_cost = tf.reduce_sum(sq_diff * y_true_mask_exp, axis=[2, 3]) 
    
    num_valid_people = tf.reduce_sum(y_true_mask_exp[:, 0, :, :], axis=[1, 2]) + 1e-6
    coords_cost_norm = coords_cost / tf.expand_dims(num_valid_people, axis=-1)

    #bce = tf.keras.backend.binary_crossentropy(y_true_mask_exp, y_pred_mask_perm)
    y_pred_mask_perm_safe = tf.clip_by_value(y_pred_mask_perm, 1e-7, 1.0 - 1e-7)
    bce = tf.keras.backend.binary_crossentropy(y_true_mask_exp, y_pred_mask_perm_safe)
    mask_cost_norm = tf.reduce_mean(bce, axis=[2, 3]) 

    total_cost = coords_cost_norm + (1.5 * mask_cost_norm) 
    return tf.reduce_min(total_cost, axis=1) 

def hungarian_rmse_metres(y_true, y_pred):
    y_true_coords = tf.reshape(y_true[:, :8], (-1, 4, 2))
    y_pred_coords = tf.reshape(y_pred[:, :8], (-1, 4, 2))
    y_true_mask = tf.reshape(y_true[:, 8:], (-1, 4, 1))

    y_pred_coords_perm = tf.gather(y_pred_coords, PERM_INDICES, axis=1)
    y_true_coords_exp = tf.expand_dims(y_true_coords, 1)
    y_true_mask_exp = tf.expand_dims(y_true_mask, 1)

    sq_diff = tf.square(y_true_coords_exp - y_pred_coords_perm)
    coords_cost = tf.reduce_sum(sq_diff * y_true_mask_exp, axis=[2, 3])
    
    min_coords_cost = tf.reduce_min(coords_cost, axis=1)
    
    num_valid_people = tf.reduce_sum(y_true_mask_exp[:, 0, :, :], axis=[1, 2]) + 1e-6
    
    return tf.sqrt(min_coords_cost / num_valid_people)

def hungarian_mask_acc(y_true, y_pred):
    y_true_mask = tf.reshape(y_true[:, 8:], (-1, 4, 1))
    y_pred_mask = tf.reshape(y_pred[:, 8:], (-1, 4, 1))
    y_pred_mask_perm = tf.gather(y_pred_mask, PERM_INDICES, axis=1)
    
    y_true_coords = tf.reshape(y_true[:, :8], (-1, 4, 2))
    y_pred_coords = tf.reshape(y_pred[:, :8], (-1, 4, 2))
    y_pred_coords_perm = tf.gather(y_pred_coords, PERM_INDICES, axis=1)
    y_true_coords_exp = tf.expand_dims(y_true_coords, 1)
    y_true_mask_exp = tf.expand_dims(y_true_mask, 1)

    sq_diff = tf.square(y_true_coords_exp - y_pred_coords_perm)
    coords_cost = tf.reduce_sum(sq_diff * y_true_mask_exp, axis=[2, 3])
    num_valid_people = tf.reduce_sum(y_true_mask_exp[:, 0, :, :], axis=[1, 2]) + 1e-6
    coords_cost_norm = coords_cost / tf.expand_dims(num_valid_people, axis=-1)
    
    #bce = tf.keras.backend.binary_crossentropy(y_true_mask_exp, y_pred_mask_perm)
    y_pred_mask_perm_safe = tf.clip_by_value(y_pred_mask_perm, 1e-7, 1.0 - 1e-7)
    bce = tf.keras.backend.binary_crossentropy(y_true_mask_exp, y_pred_mask_perm_safe)
    mask_cost_norm = tf.reduce_mean(bce, axis=[2, 3])
    
    total_cost = coords_cost_norm + (1.5 * mask_cost_norm)
    best_perm_idx = tf.argmin(total_cost, axis=1, output_type=tf.int32)
    
    batch_size = tf.shape(y_pred)[0]
    gather_nd_indices = tf.stack([tf.range(batch_size, dtype=tf.int32), best_perm_idx], axis=1)
    best_mask_pred = tf.gather_nd(y_pred_mask_perm, gather_nd_indices)
    
    return tf.reduce_mean(tf.keras.metrics.binary_accuracy(y_true_mask, best_mask_pred))

In [ ]:
# ==============================================================================
# ARCHITETTURA EEAI-NET V2 (Aggiornata per 12 Angoli)
# ==============================================================================
def build_eeai_model_v2_dbf(n_range=120, n_angle=12, n_radars=6): # <--- Impostato n_angle=12 di default
    inputs = layers.Input(shape=(n_range, n_angle, n_radars), name="radar_input")

    # Convolutional layers adattati alle mappe spaziali 2D
    x = layers.Conv2D(32, (3, 3), padding='same', activation='relu', name="conv_1")(inputs)
    x = layers.MaxPooling2D((2, 2), name="pool_1")(x) 
    
    x = layers.Conv2D(64, (3, 3), padding='same', activation='relu', name="conv_2")(x)
    x = layers.MaxPooling2D((2, 2), name="pool_2")(x) 
    
    x = layers.Conv2D(128, (3, 3), padding='same', activation='relu', name="conv_3")(x)
    x = layers.MaxPooling2D((2, 2), name="pool_3")(x) 
    
    x = layers.Conv2D(128, (3, 3), padding='same', activation='relu', name="conv_4")(x)
    x = layers.MaxPooling2D((3, 1), name="pool_4")(x) # Adattato il pooling finale alla nuova shape (12 angoli)

    x = layers.Flatten(name="flatten_features")(x)
    
    x = layers.Dense(128, activation='relu', name="features_deep1")(x)
    x = layers.Dropout(0.2, name="drop_features1")(x) 
    common_feat = layers.Dense(64, activation='relu', name="features_deep2")(x)

    coords_output = layers.Dense(8, activation='linear', name="coords_head")(common_feat)
    mask_output = layers.Dense(4, activation='sigmoid', name="mask_head")(common_feat)
    combined_output = layers.Concatenate(axis=1, name="combined_output")([coords_output, mask_output])

    return models.Model(inputs=inputs, outputs=combined_output, name="EEAI_Net_V2_DBF")

In [6]:
# ==============================================================================
# ARCHITETTURA EEAI-NET V2.1 (Aggiornata per Coordinate Normalizzate)
# ==============================================================================
def build_eeai_model_v2_dbf(n_range=120, n_angle=12, n_radars=6):
    inputs = layers.Input(shape=(n_range, n_angle, n_radars), name="radar_input")

    x = layers.Conv2D(32, (3, 3), padding='same', activation='relu', name="conv_1")(inputs)
    x = layers.MaxPooling2D((2, 2), name="pool_1")(x) 
    
    x = layers.Conv2D(64, (3, 3), padding='same', activation='relu', name="conv_2")(x)
    x = layers.MaxPooling2D((2, 2), name="pool_2")(x) 
    
    x = layers.Conv2D(128, (3, 3), padding='same', activation='relu', name="conv_3")(x)
    x = layers.MaxPooling2D((2, 2), name="pool_3")(x) 
    
    x = layers.Conv2D(128, (3, 3), padding='same', activation='relu', name="conv_4")(x)
    x = layers.MaxPooling2D((3, 1), name="pool_4")(x) 

    x = layers.Flatten(name="flatten_features")(x)
    
    x = layers.Dense(128, activation='relu', name="features_deep1")(x)
    x = layers.Dropout(0.2, name="drop_features1")(x) 
    common_feat = layers.Dense(64, activation='relu', name="features_deep2")(x)

    # CAMBIAMENTO QUI: Usiamo 'sigmoid' perché il target ora è in [0, 1]
    coords_output = layers.Dense(8, activation='sigmoid', name="coords_head")(common_feat)
    mask_output = layers.Dense(4, activation='sigmoid', name="mask_head")(common_feat)
    
    combined_output = layers.Concatenate(axis=1, name="combined_output")([coords_output, mask_output])

    return models.Model(inputs=inputs, outputs=combined_output, name="EEAI_Net_V2_DBF")

In [7]:
# ==============================================================================
# ADDESTRAMENTO MODELLO V2 DBF (FIX BUG GPU MAC M1)
# ==============================================================================
from keras.utils import Sequence

# 1. Creiamo un Generatore personalizzato per bypassare il bug di tensorflow-metal
class RadarDataGenerator(Sequence):
    def __init__(self, x_set, y_set, batch_size):
        self.x = x_set
        self.y = y_set
        self.batch_size = batch_size
        self.indices = np.arange(self.x.shape[0])
        np.random.shuffle(self.indices)

    def __len__(self):
        # Calcola quanti batch ci sono in un'epoca
        return int(np.ceil(self.x.shape[0] / float(self.batch_size)))

    def __getitem__(self, idx):
        # Estrae solo i 32 frame richiesti per questo step
        inds = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
        batch_x = self.x[inds]
        batch_y = self.y[inds]
        return batch_x, batch_y

    def on_epoch_end(self):
        # Mescola i dati alla fine di ogni epoca per evitare bias
        np.random.shuffle(self.indices)

# 2. Inizializziamo i generatori per Train e Validation
BATCH_SIZE = 32
train_gen = RadarDataGenerator(X_train, Y_train, BATCH_SIZE)
val_gen = RadarDataGenerator(X_val, Y_val, BATCH_SIZE)

# 3. Inizializzazione del nuovo modello Beamforming
model_dbf = build_eeai_model_v2_dbf()

# 4. Compilazione 
model_dbf.compile(
    optimizer='adam',
    loss=hungarian_total_loss, 
    metrics=[hungarian_rmse_metres, hungarian_mask_acc]
)

checkpoint_dbf = ModelCheckpoint("eeai_best_model_romano_v2_dbf.keras", monitor="val_loss", save_best_only=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
early_stop = EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1)

EPOCHS = 50 

print("\n--- INIZIO ADDESTRAMENTO V2 DBF (CON GENERATORE PER MAC M1) ---")
# Usiamo i generatori nel fit invece degli array diretti!
history_dbf = model_dbf.fit(
    train_gen,                
    validation_data=val_gen,  
    epochs=EPOCHS,
    callbacks=[checkpoint_dbf, reduce_lr, early_stop], 
    verbose=1
)
print("--- ADDESTRAMENTO COMPLETATO ---")


--- INIZIO ADDESTRAMENTO V2 DBF (CON GENERATORE PER MAC M1) ---
Epoch 1/50


/Users/davidepellegrino/Desktop/POLIMI/EEAI/Project/ProgettoEEAI/uwb-person-localization-tinyml/venv/lib/python3.12/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


4453/4454 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - hungarian_mask_acc: 0.9068 - hungarian_rmse_metres: 0.1745 - loss: 0.3423
Epoch 1: val_loss improved from None to 1.69922, saving model to eeai_best_model_romano_v2_dbf.keras

Epoch 1: finished saving model to eeai_best_model_romano_v2_dbf.keras
4454/4454 ━━━━━━━━━━━━━━━━━━━━ 164s 37ms/step - hungarian_mask_acc: 0.9561 - hungarian_rmse_metres: 0.1583 - loss: 0.1886 - val_hungarian_mask_acc: 0.8052 - val_hungarian_rmse_metres: 0.1296 - val_loss: 1.6992 - learning_rate: 0.0010
Epoch 2/50
4454/4454 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - hungarian_mask_acc: 0.9845 - hungarian_rmse_metres: 0.1328 - loss: 0.0859
Epoch 2: val_loss improved from 1.69922 to 1.20202, saving model to eeai_best_model_romano_v2_dbf.keras

Epoch 2: finished saving model to eeai_best_model_romano_v2_dbf.keras
4454/4454 ━━━━━━━━━━━━━━━━━━━━ 205s 46ms/step - hungarian_mask_acc: 0.9857 - hungarian_rmse_metres: 0.1296 - loss: 0.0794 - val_hungarian_mask_acc: 0.8308 - val_hungari

In [8]:
# ==============================================================================
# MODEL SUMMARY PER EMBEDDED
# ==============================================================================

def embedded_summary(model, input_shape=(120, 12, 6)):
    # 2. Calcola i parametri statici (Flash)
    total_params = model.count_params()
    estimated_flash_kb = (total_params * 4) / 1024
    
   # 3. Calcola il picco di memoria dinamica (SRAM/Tensor Arena)
    max_layer_ram_kb = 0
    for layer in model.layers:
        if layer.__class__.__name__ == 'InputLayer' or not hasattr(layer, 'output_shape'):
            continue
            
        output_shape = layer.output_shape
        if isinstance(output_shape, list):
            num_elements = sum([np.prod([dim for dim in shape[1:] if dim is not None]) for shape in output_shape])
        else:
            num_elements = np.prod([dim for dim in output_shape[1:] if dim is not None])
            
        layer_ram_kb = (num_elements * 4) / 1024
        if layer_ram_kb > max_layer_ram_kb:
            max_layer_ram_kb = layer_ram_kb

    input_elements = np.prod(input_shape)
    input_ram_kb = (input_elements * 4) / 1024
    peak_arena_kb = input_ram_kb + max_layer_ram_kb

    # 4. Stampa il verdetto 
    print("============================================")
    print("   REPORT REQUISITI HARDWARE (STIMA FLOAT32)   ")
    print("============================================")
    print(f" Memoria FLASH stimata : {estimated_flash_kb:.2f} KB  (Limite : < 800 KB)")
    print(f" Memoria SRAM stimata  : ~{peak_arena_kb:.2f} KB (Limite : < 400 KB)")
    print(" Operazioni Ricorrenti : ASSENTI (RNN/LSTM/GRU non rilevate)")
    print(" Nota sulla Quantizz.  : Raccomandata INT8 per ESP32-S3 (ridurrà la RAM di ~4x)")
    print("============================================\n")

embedded_summary(model_dbf)
# model_dbf.summary()

   REPORT REQUISITI HARDWARE (STIMA FLOAT32)   
 Memoria FLASH stimata : 1299.92 KB  (Limite : < 800 KB)
 Memoria SRAM stimata  : ~33.75 KB (Limite : < 400 KB)
 Operazioni Ricorrenti : ASSENTI (RNN/LSTM/GRU non rilevate)
 Nota sulla Quantizz.  : Raccomandata INT8 per ESP32-S3 (ridurrà la RAM di ~4x)



In [10]:
# ==============================================================================
# VISUALIZZATORE V4 (Compatibile con la nuova V2 DBF)
# ==============================================================================

file_target = "dataset/window_000005.npz"
if not os.path.exists(file_target):
    print(f"ERRORE: Non trovo il file {file_target}")
else:
    data = np.load(file_target)
    raw_iq = data['radar_cir_iq'] 
    gt_coords = data['people_xy'] 
    gt_mask = data['people_mask'] 
    T = raw_iq.shape[0]

    print("Elaborazione filtri DBF e previsioni in corso (V10)...")
    
    # 1. Ricostruzione del Range-Angle Map per il file live
    num_angles = 12
    theta = np.linspace(-np.pi/3, np.pi/3, num_angles)
    n_antennas_array = np.arange(3)
    W = np.exp(-1j * np.pi * np.outer(np.sin(theta), n_antennas_array))
    
    complex_iq = raw_iq[..., 0] + 1j * raw_iq[..., 1]
    iq_transposed = np.transpose(complex_iq, (0, 1, 3, 2))
    dbf_result = np.dot(iq_transposed, W.T)
    mag = np.abs(dbf_result)
    mag_img = np.transpose(mag, (0, 2, 3, 1)).astype(np.float32)

    # 2. EMA
    decluttered = np.zeros_like(mag_img)
    bg = np.copy(mag_img[0])
    alpha = 0.20
    for t in range(T):
        bg = alpha * mag_img[t] + (1 - alpha) * bg
        decluttered[t] = np.abs(mag_img[t] - bg)

    # 3. Normalizzazione
    max_val = np.max(decluttered)
    min_val = np.min(decluttered)
    decluttered = (decluttered - min_val) / (max_val - min_val + 1e-8)

    print("Caricamento dei pesi migliori dal file .keras ...")
    model_dbf = load_model(
        "eeai_best_model_romano_v2_dbf.keras",
        custom_objects={
            "hungarian_total_loss": hungarian_total_loss,
            "hungarian_rmse_metres": hungarian_rmse_metres, 
            "hungarian_mask_acc": hungarian_mask_acc
        }
    )

    preds = model_dbf.predict(decluttered, verbose=0)
    
    p_coords = preds[:, :8].reshape(T, 4, 2)
    p_mask = preds[:, 8:]
    
    print("Dati pronti! Inizializzazione Radar...")

    out = widgets.Output() 

    def draw_frame(frame_idx, soglia):
        with out:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(4.8, 7.2))
            ax.set_xlim(-0.5, 5.3); ax.set_ylim(-0.5, 7.7)
            ax.grid(True, linestyle=':', alpha=0.6)
            
            ax.set_title(f"Radar DBF V2 | Frame: {frame_idx}/{T-1}", fontsize=14, fontweight='bold')

            stanza = plt.Rectangle((0, 0), 4.8, 7.2, linewidth=3, edgecolor='navy', facecolor='whitesmoke')
            ax.add_patch(stanza)

            for i in range(4):
                is_present = bool(gt_mask[frame_idx, i] > 0.5)
                if is_present:
                    rx, ry = gt_coords[frame_idx, i]
                    ax.scatter(rx, ry, c='limegreen', s=120, edgecolors='black', marker='o', label='REALE (GT)' if i==0 else "")
                    ax.text(rx, ry + 0.2, f"P{i+1}", color='darkgreen', fontweight='bold', ha='center')

                conf = float(p_mask[frame_idx, i])
                if conf >= soglia:
                    px, py = p_coords[frame_idx, i]
                    alpha_val = max(0.3, conf)
                    ax.scatter(px, py, c='red', s=100, marker='X', edgecolors='darkred', alpha=alpha_val, label='PREDETTO' if i==0 else "")
                    ax.text(px, py - 0.3, f"{conf*100:.0f}%", color='red', fontsize=10, ha='center', fontweight='bold')

            handles, labels = ax.get_legend_handles_labels()
            by_label = dict(zip(labels, handles))
            if by_label:
                ax.legend(by_label.values(), by_label.keys(), loc='upper right', frameon=True, shadow=True)

            plt.xlabel("X (Metri)"); plt.ylabel("Y (Metri)")
            plt.tight_layout(); plt.show()

    slider_frame = widgets.IntSlider(value=500, min=10, max=T-1, step=1, description='Frame:')
    slider_soglia = widgets.FloatSlider(value=0.50, min=0.1, max=0.99, step=0.05, description='Soglia:')

    def on_change(change):
        draw_frame(slider_frame.value, slider_soglia.value)

    slider_frame.observe(on_change, names='value')
    slider_soglia.observe(on_change, names='value')

    controls = widgets.VBox([slider_frame, slider_soglia])
    controls.layout.margin = '20px 20px 20px 0px' 
    ui = widgets.HBox([controls, out])
    
    display(ui)
    draw_frame(slider_frame.value, slider_soglia.value)

Elaborazione filtri DBF e previsioni in corso (V10)...
Caricamento dei pesi migliori dal file .keras ...
Dati pronti! Inizializzazione Radar...


In [ ]:
# ==============================================================================
# VISUALIZZATORE V4.1 (Allineato a EMA Complesso e Denormalizzazione)
# ==============================================================================

file_target = "dataset/window_000005.npz"
if not os.path.exists(file_target):
    print(f"ERRORE: Non trovo il file {file_target}")
else:
    data = np.load(file_target)
    raw_iq = data['radar_cir_iq'] 
    gt_coords = data['people_xy'] # Queste NON sono normalizzate (Ground Truth originale)
    gt_mask = data['people_mask'] 
    T = raw_iq.shape[0]

    print("Elaborazione filtri DBF ed EMA in corso...")
    
    # 1. DBF Setup
    num_angles = 12
    d_lambda = 0.5
    theta = np.linspace(-np.pi/3, np.pi/3, num_angles)
    n_antennas_array = np.arange(3)
    W = np.exp(-1j * 2 * np.pi * d_lambda * np.outer(np.sin(theta), n_antennas_array))
    
    # 2. Complessi ed EMA
    complex_iq = raw_iq[..., 0] + 1j * raw_iq[..., 1]
    decluttered_iq = np.zeros_like(complex_iq)
    bg = np.copy(complex_iq[0])
    alpha = 0.20
    for t in range(T):
        bg = alpha * complex_iq[t] + (1 - alpha) * bg
        decluttered_iq[t] = complex_iq[t] - bg
        
    # 3. DBF e Magnitudo
    iq_transposed = np.transpose(decluttered_iq, (0, 1, 3, 2))
    dbf_result = np.dot(iq_transposed, W.T)
    mag = np.abs(dbf_result)
    mag_img = np.transpose(mag, (0, 2, 3, 1)).astype(np.float32)

    # 4. Normalizzazione input
    max_val = np.max(mag_img)
    min_val = np.min(mag_img)
    mag_img = (mag_img - min_val) / (max_val - min_val + 1e-8)

    print("Caricamento dei pesi migliori dal file .keras ...")
    model_dbf = load_model(
        "eeai_best_model_romano_v2_dbf.keras",
        custom_objects={
            "hungarian_total_loss": hungarian_total_loss,
            "hungarian_rmse_metres": hungarian_rmse_metres, 
            "hungarian_mask_acc": hungarian_mask_acc
        }
    )

    preds = model_dbf.predict(mag_img, verbose=0)
    
    p_coords = preds[:, :8].reshape(T, 4, 2)
    p_mask = preds[:, 8:]
    
    print("Dati pronti! Inizializzazione Radar...")

    out = widgets.Output() 

    def draw_frame(frame_idx, soglia):
        with out:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(4.8, 7.2))
            ax.set_xlim(-0.5, 5.3); ax.set_ylim(-0.5, 7.7)
            ax.grid(True, linestyle=':', alpha=0.6)
            
            ax.set_title(f"Radar DBF V2.1 | Frame: {frame_idx}/{T-1}", fontsize=14, fontweight='bold')

            stanza = plt.Rectangle((0, 0), 4.8, 7.2, linewidth=3, edgecolor='navy', facecolor='whitesmoke')
            ax.add_patch(stanza)

            for i in range(4):
                is_present = bool(gt_mask[frame_idx, i] > 0.5)
                if is_present:
                    rx, ry = gt_coords[frame_idx, i]
                    ax.scatter(rx, ry, c='limegreen', s=120, edgecolors='black', marker='o', label='REALE (GT)' if i==0 else "")
                    ax.text(rx, ry + 0.2, f"P{i+1}", color='darkgreen', fontweight='bold', ha='center')

                conf = float(p_mask[frame_idx, i])
                if conf >= soglia:
                    px, py = p_coords[frame_idx, i]
                    
                    # --- DE-NORMALIZZAZIONE PER IL PLOT ---
                    px = px * 4.8
                    py = py * 7.2
                    
                    alpha_val = max(0.3, conf)
                    ax.scatter(px, py, c='red', s=100, marker='X', edgecolors='darkred', alpha=alpha_val, label='PREDETTO' if i==0 else "")
                    ax.text(px, py - 0.3, f"{conf*100:.0f}%", color='red', fontsize=10, ha='center', fontweight='bold')

            handles, labels = ax.get_legend_handles_labels()
            by_label = dict(zip(labels, handles))
            if by_label:
                ax.legend(by_label.values(), by_label.keys(), loc='upper right', frameon=True, shadow=True)

            plt.xlabel("X (Metri)"); plt.ylabel("Y (Metri)")
            plt.tight_layout(); plt.show()

    slider_frame = widgets.IntSlider(value=500, min=10, max=T-1, step=1, description='Frame:')
    slider_soglia = widgets.FloatSlider(value=0.50, min=0.1, max=0.99, step=0.05, description='Soglia:')

    def on_change(change):
        draw_frame(slider_frame.value, slider_soglia.value)

    slider_frame.observe(on_change, names='value')
    slider_soglia.observe(on_change, names='value')

    controls = widgets.VBox([slider_frame, slider_soglia])
    controls.layout.margin = '20px 20px 20px 0px' 
    ui = widgets.HBox([controls, out])
    
    display(ui)
    draw_frame(slider_frame.value, slider_soglia.value)

Elaborazione filtri DBF ed EMA in corso...
Caricamento dei pesi migliori dal file .keras ...
Dati pronti! Inizializzazione Radar...
